In [1]:
import os
import json
import pandas as pd

In [ ]:
# Tratamento json
with open('data/reviews.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

textos = []
labels = []

for paper in dados['paper']:
    for review in paper['review']:
        if review['text'].strip():
            textos.append(review['text'].strip())
            labels.append(int(review['evaluation']))

df = pd.DataFrame({
    'texto': textos,
    'avaliacao': labels
})

df.to_csv('dataset.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Análise inicial
df = pd.read_csv('data/dataset.csv')
df.head(10)

,texto,avaliacao
0,- El artículo aborda un problema contingente y...,1
1,El artículo presenta recomendaciones prácticas...,1
2,- El tema es muy interesante y puede ser de mu...,1
3,Se explica en forma ordenada y didáctica una e...,2
4,Los autores describen una metodología para des...,2
5,Este trabajo propone un nuevo enfoque basado e...,2
6,This paper aims to show new deployment alterna...,2
7,The paper is well structured. It follows a log...,0
8,Se realiza un trabajo de modelamiento de encri...,2
9,Este paper involucra el desarrollo de una apli...,-2


In [ ]:
# Classificar pra analisar quantidade de avaliações por tipo
def classificar(nota):
    if nota in [-2, -1]:
        return 'negativo'
    elif nota == 0:
        return 'neutro'
    else:
        return 'positivo'
df['sentimento'] = df['avaliacao'].apply(classificar)

total = len(df)

for sentimento, qtd in df['sentimento'].value_counts().items():
    pct = (qtd / total) * 100
    print(f"{sentimento}: {qtd} avaliações ({pct:.1f}%)")


positivo: 192 avaliações (48.1%)
negativo: 145 avaliações (36.3%)
neutro: 62 avaliações (15.5%)


In [20]:
# Tirar neutras
df_binario = df[df['avaliacao'] != 0].copy()

df_binario['avaliacao'].value_counts().sort_index()

avaliacao
-2     86
-1     59
 1     86
 2    106
Name: count, dtype: int64

In [23]:
# Criar label binário: 
# notas > 0 viram 1 (positivo), notas < 0 viram 0 (negativo)
df_binario['label'] = (df_binario['avaliacao'] > 0).astype(int)

df_binario.head(10)

,texto,avaliacao,sentimento,label
0,- El artículo aborda un problema contingente y...,1,positivo,1
1,El artículo presenta recomendaciones prácticas...,1,positivo,1
2,- El tema es muy interesante y puede ser de mu...,1,positivo,1
3,Se explica en forma ordenada y didáctica una e...,2,positivo,1
4,Los autores describen una metodología para des...,2,positivo,1
5,Este trabajo propone un nuevo enfoque basado e...,2,positivo,1
6,This paper aims to show new deployment alterna...,2,positivo,1
8,Se realiza un trabajo de modelamiento de encri...,2,positivo,1
9,Este paper involucra el desarrollo de una apli...,-2,negativo,0
10,Novedosa propuesta. Bien estructurada. Correct...,2,positivo,1


In [25]:
df_binario.to_csv('dataset_tratado.csv', index=False, encoding='utf-8-sig')

In [2]:
# (voltando pra etapa de tratamento)
# Remover 47 positivos aleatóriamente 
df = pd.read_csv('data/dataset_tratado.csv')

In [4]:
df_neg = df[df['label'] == 0]
df_pos = df[df['label'] == 1]
df_pos_145 = df_pos.sample(n=len(df_neg), random_state=1)

df_balanceado = pd.concat([df_neg, df_pos_145]).sample(frac=1, random_state=42).reset_index(drop=True)

df_balanceado.to_csv('data/dataset_balanceado.csv', index=False, encoding='utf-8-sig')

print(f"Negativos: {(df_balanceado['label'] == 0).sum()}")
print(f"Positivos: {(df_balanceado['label'] == 1).sum()}")

Negativos: 145
Positivos: 145
